# PolitCheck – Datenbank Explorer

Dieses Notebook liest Daten aus beiden SQLite-Datenbanken via SQL aus.

| Datenbank | Pfad | Inhalt |
|-----------|------|--------|
| Rohdaten  | `data/plenarprotokolle.db` | API-Metadaten aller Plenarprotokolle |
| Analyse   | `data/politcheck.db`       | LLM-extrahierte Aussagen + Politiker-Stats |

In [ ]:
import sqlite3
import pandas as pd
import json
from pathlib import Path

# Pfade relativ zum Notebook (eda/ -> data/)
DB_RAW      = Path('../../data/plenarprotokolle.db')
DB_ANALYSE  = Path('../../data/politcheck.db')

def query(db_path: Path, sql: str, params=()) -> pd.DataFrame:
    """Führt eine SQL-Abfrage aus und gibt ein DataFrame zurück."""
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(sql, conn, params=params)

print('Rohdaten-DB existiert:', DB_RAW.exists())
print('Analyse-DB existiert: ', DB_ANALYSE.exists())

---
## 1. Rohdaten-DB – `plenarprotokolle.db`

In [ ]:
# Übersicht: Gesamtanzahl, Zeitraum
query(DB_RAW, """
    SELECT
        COUNT(*)       AS gesamt,
        MIN(datum)     AS aeltestes_datum,
        MAX(datum)     AS neuestes_datum
    FROM protokolle
""")

In [ ]:
# Protokolle pro Wahlperiode
query(DB_RAW, """
    SELECT
        wahlperiode,
        COUNT(*) AS anzahl
    FROM protokolle
    GROUP BY wahlperiode
    ORDER BY wahlperiode DESC
""")

In [ ]:
# Neueste 20 Protokolle
query(DB_RAW, """
    SELECT
        id,
        datum,
        dokumentnummer,
        titel,
        pdf_url
    FROM protokolle
    ORDER BY datum DESC
    LIMIT 20
""")

In [ ]:
# Protokolle eines bestimmten Zeitraums filtern
query(DB_RAW, """
    SELECT
        id,
        datum,
        dokumentnummer,
        titel,
        abstract
    FROM protokolle
    WHERE datum BETWEEN '2025-01-01' AND '2025-12-31'
    ORDER BY datum DESC
""")

In [ ]:
# Protokolle mit Abstract (nicht NULL)
query(DB_RAW, """
    SELECT
        COUNT(*) AS mit_abstract,
        (SELECT COUNT(*) FROM protokolle) - COUNT(*) AS ohne_abstract
    FROM protokolle
    WHERE abstract IS NOT NULL AND abstract != ''
""")

In [ ]:
# Ein einzelnes Protokoll vollständig anzeigen (inkl. raw_json)
df = query(DB_RAW, """
    SELECT * FROM protokolle
    ORDER BY datum DESC
    LIMIT 1
""")

# raw_json schön formatieren
row = df.iloc[0]
print('Titel:         ', row['titel'])
print('Datum:         ', row['datum'])
print('Dokumentnr.:   ', row['dokumentnummer'])
print('PDF-URL:       ', row['pdf_url'])
print('Abstract:      ', str(row['abstract'])[:200])
print()
print('--- raw_json (Auszug) ---')
raw = json.loads(row['raw_json'])
print(json.dumps(raw, indent=2, ensure_ascii=False)[:1000])

In [ ]:
# Protokolle nach Import-Datum (gespeichert_am) – wann wurden welche Daten importiert?
query(DB_RAW, """
    SELECT
        DATE(gespeichert_am) AS import_datum,
        COUNT(*)             AS anzahl
    FROM protokolle
    GROUP BY DATE(gespeichert_am)
    ORDER BY import_datum DESC
    LIMIT 10
""")

In [ ]:
# Protokolle pro Jahr und Monat
query(DB_RAW, """
    SELECT
        strftime('%Y', datum)       AS jahr,
        strftime('%m', datum)       AS monat,
        COUNT(*)                    AS anzahl
    FROM protokolle
    WHERE datum IS NOT NULL
    GROUP BY jahr, monat
    ORDER BY jahr DESC, monat DESC
""")

---
## 2. Analyse-DB – `politcheck.db`

In [ ]:
# Übersicht Analyse-DB
query(DB_ANALYSE, """
    SELECT
        (SELECT COUNT(*) FROM aussagen)             AS aussagen_gesamt,
        (SELECT COUNT(*) FROM verarbeitete_quellen) AS verarbeitete_protokolle
""")

In [ ]:
# Top 20 Aussagen nach Polarisierungsgrad
query(DB_ANALYSE, """
    SELECT
        politiker,
        partei,
        datum,
        polarisierungsgrad,
        thema,
        aussage
    FROM aussagen
    ORDER BY polarisierungsgrad DESC
    LIMIT 20
""")

In [ ]:
# Aussagen nach Thema gruppiert
query(DB_ANALYSE, """
    SELECT
        thema,
        COUNT(*)                              AS anzahl,
        ROUND(AVG(polarisierungsgrad), 1)     AS avg_polarisierung,
        MAX(polarisierungsgrad)               AS max_polarisierung
    FROM aussagen
    GROUP BY thema
    ORDER BY anzahl DESC
""")

In [ ]:
# Politiker-Ranking nach durchschnittlichem Polarisierungsgrad
query(DB_ANALYSE, """
    SELECT
        politiker,
        partei,
        COUNT(*)                              AS anzahl_aussagen,
        ROUND(AVG(polarisierungsgrad), 1)     AS avg_polarisierung,
        MAX(polarisierungsgrad)               AS max_polarisierung
    FROM aussagen
    GROUP BY politiker, partei
    HAVING anzahl_aussagen >= 2
    ORDER BY avg_polarisierung DESC
    LIMIT 20
""")

In [ ]:
# Aussagen nach Partei filtern (z.B. AfD)
partei_filter = 'AfD'  # <-- hier anpassen

query(DB_ANALYSE, """
    SELECT
        politiker,
        datum,
        polarisierungsgrad,
        thema,
        aussage
    FROM aussagen
    WHERE partei = ?
    ORDER BY polarisierungsgrad DESC
    LIMIT 20
""", params=(partei_filter,))

In [ ]:
# Aussagen nach Thema filtern
thema_filter = 'Migration'  # <-- hier anpassen

query(DB_ANALYSE, """
    SELECT
        politiker,
        partei,
        datum,
        polarisierungsgrad,
        aussage,
        polarisierungsbegruendung
    FROM aussagen
    WHERE thema = ?
    ORDER BY polarisierungsgrad DESC
""", params=(thema_filter,))

In [ ]:
# Verteilung der Polarisierungsgrade (Histogramm-Daten)
query(DB_ANALYSE, """
    SELECT
        polarisierungsgrad,
        COUNT(*) AS anzahl
    FROM aussagen
    GROUP BY polarisierungsgrad
    ORDER BY polarisierungsgrad DESC
""")

In [ ]:
# Verarbeitete Quellen – welche Protokoll-IDs wurden bereits durch Workflow 2 verarbeitet?
query(DB_ANALYSE, """
    SELECT
        quelle_id,
        quelle_typ,
        verarbeitet_am
    FROM verarbeitete_quellen
    ORDER BY verarbeitet_am DESC
    LIMIT 20
""")

---
## 3. Cross-DB: Welche Protokolle wurden noch NICHT verarbeitet?

In [ ]:
# Da zwei separate DBs: Python-seitig joinen
with sqlite3.connect(DB_RAW) as conn_raw:
    alle_ids = set(
        r[0] for r in conn_raw.execute('SELECT id FROM protokolle').fetchall()
    )

with sqlite3.connect(DB_ANALYSE) as conn_analyse:
    verarbeitete_ids = set(
        r[0] for r in conn_analyse.execute('SELECT quelle_id FROM verarbeitete_quellen').fetchall()
    )

nicht_verarbeitet = alle_ids - verarbeitete_ids

print(f'Protokolle in Rohdaten-DB total:       {len(alle_ids)}')
print(f'Davon bereits durch Workflow 2 extrahiert: {len(verarbeitete_ids)}')
print(f'Noch ausstehend (Workflow 2):              {len(nicht_verarbeitet)}')

In [ ]:
# Details zu noch nicht verarbeiteten Protokollen (neueste zuerst)
if nicht_verarbeitet:
    placeholders = ','.join('?' * len(nicht_verarbeitet))
    with sqlite3.connect(DB_RAW) as conn:
        df = pd.read_sql_query(
            f'SELECT id, datum, dokumentnummer, titel FROM protokolle WHERE id IN ({placeholders}) ORDER BY datum DESC LIMIT 20',
            conn,
            params=list(nicht_verarbeitet)
        )
    df
else:
    print('Alle Protokolle wurden bereits verarbeitet.')